# 🏎️ F1 Driver–Circuit Compatibility
## Notebook 01 — Data Collection

**What this notebook does:**
- Fetches all F1 race data (2019–2025) using the FastF1 API
- Saves progress after every season (crash-safe)
- Outputs `laps_raw.csv`, `results_raw.csv`, `schedule_raw.csv`

**Output:** `data/raw/`

> ⏱️ First run takes ~45–90 min. Subsequent runs use cache (~2 min).

In [ ]:
# Install dependencies (run once)
%pip install fastf1 --quiet

In [ ]:
import fastf1
import pandas as pd
import numpy as np
import os
import time
import warnings
warnings.filterwarnings('ignore')

print(f"✅ FastF1 version: {fastf1.__version__}")

## Step 1 — Configure Paths

All paths are relative to this notebook's location. The project root is one level up.

In [ ]:
# Resolve project root relative to this notebook
NOTEBOOK_DIR = os.path.dirname(os.path.abspath('__file__'))
PROJECT_ROOT = os.path.abspath(os.path.join(NOTEBOOK_DIR, '..'))

PATHS = {
    'raw':       os.path.join(PROJECT_ROOT, 'data', 'raw'),
    'processed': os.path.join(PROJECT_ROOT, 'data', 'processed'),
    'cache':     os.path.join(PROJECT_ROOT, 'data', 'cache'),
    'figures':   os.path.join(PROJECT_ROOT, 'figures'),
    'models':    os.path.join(PROJECT_ROOT, 'models'),
}

for name, path in PATHS.items():
    os.makedirs(path, exist_ok=True)
    print(f"✅ {name}: {path}")

fastf1.Cache.enable_cache(PATHS['cache'])
print(f"\n✅ Cache enabled at: {PATHS['cache']}")

## Step 2 — Configuration

In [ ]:
SEASONS               = [2019, 2020, 2021, 2022, 2023, 2024, 2025]
SLEEP_BETWEEN_RACES   = 5    # seconds between each race load
SLEEP_ON_RATE_LIMIT   = 90   # seconds to pause when rate limit is detected
MAX_RETRIES           = 3    # retries per race before giving up

LAP_COLS = [
    'Season', 'RoundNumber', 'CircuitName', 'EventName',
    'Driver', 'Team',
    'LapNumber', 'LapTime',
    'Sector1Time', 'Sector2Time', 'Sector3Time',
    'Compound', 'TyreLife', 'Stint',
    'PitInTime', 'PitOutTime',
    'TrackStatus', 'IsPersonalBest'
]

RESULT_COLS = [
    'Season', 'RoundNumber', 'CircuitName', 'EventName',
    'Abbreviation', 'FullName', 'TeamName',
    'GridPosition', 'Position',
    'Points', 'Status'
]

print("✅ Config ready.")
print(f"   Seasons: {SEASONS}")
print(f"   Delay between races: {SLEEP_BETWEEN_RACES}s | Rate limit pause: {SLEEP_ON_RATE_LIMIT}s")

## Step 3 — Collect Schedule

In [ ]:
all_schedules = []

for season in SEASONS:
    try:
        schedule = fastf1.get_event_schedule(season, include_testing=False)
        schedule['Season'] = season
        all_schedules.append(schedule)
        print(f"✅ {season}: {len(schedule)} events")
        time.sleep(2)
    except Exception as e:
        print(f"❌ {season}: {e}")

schedule_df = pd.concat(all_schedules, ignore_index=True)
schedule_df.to_csv(os.path.join(PATHS['raw'], 'schedule_raw.csv'), index=False)
print(f"\n💾 schedule_raw.csv saved — {len(schedule_df)} rows")

## Step 4 — Helper Functions

In [ ]:
def is_rate_limit_error(error_msg):
    keywords = ['500 calls', 'rate limit', 'too many requests', '429', 'ratelimit']
    return any(k.lower() in str(error_msg).lower() for k in keywords)


def load_session_with_retry(season, round_number, max_retries=MAX_RETRIES):
    """Load a race session with retry + rate-limit backoff."""
    for attempt in range(1, max_retries + 1):
        try:
            session = fastf1.get_session(season, round_number, 'R')
            session.load(telemetry=False, weather=False, messages=False)

            laps = session.laps.copy()
            laps['Season']      = season
            laps['RoundNumber'] = round_number
            laps['CircuitName'] = session.event['Location']
            laps['EventName']   = session.event['EventName']
            laps = laps[[c for c in LAP_COLS if c in laps.columns]]

            results = session.results.copy()
            results['Season']      = season
            results['RoundNumber'] = round_number
            results['CircuitName'] = session.event['Location']
            results['EventName']   = session.event['EventName']
            results = results[[c for c in RESULT_COLS if c in results.columns]]

            return laps, results

        except Exception as e:
            if is_rate_limit_error(str(e)):
                print(f"     ⚠️  Rate limit! Pausing {SLEEP_ON_RATE_LIMIT}s... (attempt {attempt}/{max_retries})")
                time.sleep(SLEEP_ON_RATE_LIMIT)
            else:
                print(f"     ⚠️  Attempt {attempt}/{max_retries} failed: {str(e)[:80]}")
                if attempt < max_retries:
                    time.sleep(10)

    return None, None


def load_existing_csv(path):
    if os.path.exists(path):
        df = pd.read_csv(path)
        print(f"   📂 Loaded existing: {os.path.basename(path)} ({len(df):,} rows)")
        return df
    return pd.DataFrame()


print("✅ Helper functions ready.")

## Step 5 — Main Collection Loop

> **If this crashes:** Re-run this cell. It automatically skips completed seasons.

In [ ]:
laps_path    = os.path.join(PATHS['raw'], 'laps_raw.csv')
results_path = os.path.join(PATHS['raw'], 'results_raw.csv')

print("🔍 Checking for existing data...")
laps_master    = load_existing_csv(laps_path)
results_master = load_existing_csv(results_path)

if not laps_master.empty and 'Season' in laps_master.columns:
    completed_seasons = set(laps_master['Season'].unique())
    print(f"   ✅ Already collected: {sorted(completed_seasons)}")
else:
    completed_seasons = set()
    print("   📭 Starting fresh.")

seasons_to_run = [s for s in SEASONS if s not in completed_seasons]
print(f"\n📋 Seasons remaining: {seasons_to_run}")
if not seasons_to_run:
    print("🎉 All seasons already collected!")

all_failed = []

for season in seasons_to_run:
    print(f"\n{'='*55}")
    print(f"📅 SEASON {season}")
    print(f"{'='*55}")

    season_laps, season_results, season_failed = [], [], []

    try:
        schedule = fastf1.get_event_schedule(season, include_testing=False)
        time.sleep(2)
    except Exception as e:
        print(f"  ❌ Could not fetch schedule: {e}")
        continue

    total_races = len(schedule)

    for idx, (_, event) in enumerate(schedule.iterrows(), start=1):
        round_num    = event['RoundNumber']
        circuit_name = event.get('Location', 'Unknown')

        print(f"  [{idx:02d}/{total_races:02d}] {circuit_name} ... ", end="", flush=True)

        laps, results = load_session_with_retry(season, round_num)

        if laps is not None:
            season_laps.append(laps)
            season_results.append(results)
            print(f"✅ {len(laps)} laps | {len(results)} drivers")
        else:
            print(f"❌ Skipped")
            season_failed.append({'season': season, 'round': round_num, 'circuit': circuit_name})

        time.sleep(SLEEP_BETWEEN_RACES)

    if season_laps:
        laps_master    = pd.concat([laps_master,    pd.concat(season_laps,    ignore_index=True)], ignore_index=True)
        results_master = pd.concat([results_master, pd.concat(season_results, ignore_index=True)], ignore_index=True)
        laps_master.to_csv(laps_path,    index=False)
        results_master.to_csv(results_path, index=False)
        print(f"\n  💾 Season {season} saved — {len(season_laps)} races | {len(season_failed)} failed")
        print(f"     Running total: {len(laps_master):,} lap rows")

    all_failed.extend(season_failed)

    if season != seasons_to_run[-1]:
        print(f"  ⏸️  Pausing 15s before next season...")
        time.sleep(15)

print(f"\n{'='*55}")
print("🏁 Collection complete!")
print(f"   Total laps rows:    {len(laps_master):,}")
print(f"   Total results rows: {len(results_master):,}")
print(f"   Total failed races: {len(all_failed)}")

## Step 6 — Review Failures

In [ ]:
if all_failed:
    print(f"⚠️  {len(all_failed)} races failed to load:")
    for f in all_failed:
        print(f"   {f['season']} Round {f['round']:02d} — {f['circuit']}")
    print("\n💡 Re-run Step 5 — completed seasons will be skipped automatically.")
else:
    print("✅ No failed races!")

## Step 7 — Sanity Checks

In [ ]:
laps_check    = pd.read_csv(laps_path)
results_check = pd.read_csv(results_path)

print("📊 Final file sizes:")
print(f"   laps_raw:    {laps_check.shape[0]:,} rows × {laps_check.shape[1]} cols")
print(f"   results_raw: {results_check.shape[0]:,} rows × {results_check.shape[1]} cols")

print("\n📅 Races per season:")
print(results_check.groupby('Season')['RoundNumber'].nunique().to_string())

print("\n🏎️  Drivers per season:")
print(results_check.groupby('Season')['Abbreviation'].nunique().to_string())

In [ ]:
laps_check.head(5)